# Coupled cluster theory

Companion notebook to Chapter 10 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here; the code is
the same as in `BookManybody/BookMaterial/Programs/coupledcluster.py`.

The ansatz is a single change from configuration interaction:

$$|\Psi\rangle = e^{\hat T}|\Phi_0\rangle,
\qquad \hat T = \hat T_1 + \hat T_2 + \cdots$$

Requiring $\overline H = e^{-\hat T}\hat H e^{\hat T}$ to have the reference
as an eigenstate gives

$$\Delta E = \langle\Phi_0|\overline H_N|\Phi_0\rangle,\qquad
  0 = \langle\Phi_i^a|\overline H_N|\Phi_0\rangle,\qquad
  0 = \langle\Phi_{ij}^{ab}|\overline H_N|\Phi_0\rangle.$$

Contents:

1. Validating the solver
2. The pairing model
3. What coupled cluster contains, order by order
4. When singles matter
5. Size extensivity
6. Everything against everything

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import coupledcluster as cc

np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. Validating the solver

Four checks: the models must reproduce the exact energies of Chapters 4 and
7; the singles must vanish for the pairing model; the first iteration must be
MP2; and the Hartree-Fock rotation must leave the exact energy alone while
making $f_{ia}$ vanish.

In [ ]:
cc.demo_validation()

## 2. The pairing model

Four doubly degenerate levels, four particles.  The reference energy is the
Hartree-Fock energy $2-g$ of Chapter 6, so everything below is correlation
energy.

In [ ]:
cc.demo_pairing()

In [ ]:
gs = np.linspace(0.1, 2.2, 22)
ref, mp2, ccd_e, exact = [], [], [], []
for g in gs:
    h, v, N = cc.pairing_model(g=g)
    fk = cc.fock_matrix(h, v, N)
    e0 = cc.reference_energy(h, v, N)
    ref.append(e0)
    mp2.append(e0 + cc.mp2_energy(fk, v, N))
    ccd_e.append(e0 + cc.ccd(fk, v, N)["energy"])
    exact.append(cc.fci_energy(h, v, N)[0])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(gs, exact, "k-o", ms=3, label="exact (FCI)")
ax[0].plot(gs, ref, "C7--", label="Hartree-Fock")
ax[0].plot(gs, mp2, "C0-", label="MP2")
ax[0].plot(gs, ccd_e, "C3-", label="CCD")
ax[0].set_xlabel("$g$"); ax[0].set_ylabel("ground-state energy")
ax[0].legend()
ax[1].semilogy(gs, np.abs(np.array(mp2) - np.array(exact)), "C0-", label="MP2")
ax[1].semilogy(gs, np.abs(np.array(ccd_e) - np.array(exact)), "C3-",
               label="CCD")
ax[1].set_xlabel("$g$"); ax[1].set_ylabel("|error|")
ax[1].set_title("CCD is exact to four parts in a million at weak coupling")
ax[1].legend()
fig.tight_layout(); plt.show()

## 3. What coupled cluster contains, order by order

The CCD equation is exactly quadratic in the amplitudes, so it splits as
$R(t) = C + L[t] + Q[t,t]$ and can be expanded in powers of the interaction.
The result is Møller-Plesset perturbation theory — verified here against the
independent implementation of Chapter 9.

In [ ]:
cc.demo_orders()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for g, style in ((0.5, "-o"), (1.0, "-s"), (1.5, "-^")):
    h, v, N = cc.pairing_model(g=g)
    fk = cc.fock_matrix(h, v, N)
    e, _ = cc.ccd_order_by_order(fk, v, N, order=16)
    ax.semilogy(range(2, 18), np.abs(e), style, ms=4, label=f"$g = {g}$")
ax.set_xlabel("order in the interaction")
ax.set_ylabel(r"$|\Delta E^{(n)}|$")
ax.set_title("the perturbative content of CCD")
ax.legend(); fig.tight_layout(); plt.show()

The expansion converges back onto the CCD energy.  Solving the nonlinear
equation reaches the same answer in twenty-odd iterations without ever
referring to the order structure — that is what "infinite resummation" means
in practice.

## 4. When singles matter

The particle-hole term of Chapter 7 breaks pairs, so the reference couples to
$1p$-$1h$ states and the singles amplitudes are no longer zero.  We run CCSD
twice: on the bare oscillator reference and on the Hartree-Fock reference,
where Brillouin's theorem makes the singles start one order later.

In [ ]:
cc.demo_ccsd()

In [ ]:
fs = np.linspace(0.0, 0.5, 11)
g = 0.5
d_e, s_e, ex_e, t1max = [], [], [], []
for f in fs:
    h, v, N = cc.pairing_ph_model(g=g, f=f)
    fk = cc.fock_matrix(h, v, N)
    e0 = cc.reference_energy(h, v, N)
    d_e.append(e0 + cc.ccd(fk, v, N, mixing=0.3)["energy"])
    s = cc.ccsd(fk, v, N, mixing=0.3)
    s_e.append(e0 + s["energy"]); t1max.append(np.abs(s["t1"]).max())
    ex_e.append(cc.fci_energy(h, v, N)[0])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(fs, ex_e, "k-o", ms=3, label="exact")
ax[0].plot(fs, d_e, "C0-", label="CCD")
ax[0].plot(fs, s_e, "C3-", label="CCSD")
ax[0].set_xlabel("$f$"); ax[0].set_ylabel("ground-state energy")
ax[0].set_title(f"$g = {g}$"); ax[0].legend()
ax[1].plot(fs, t1max, "C2-o", ms=3)
ax[1].set_xlabel("$f$"); ax[1].set_ylabel(r"$\max|t_1|$")
ax[1].set_title("the singles grow with the pair-breaking term")
fig.tight_layout(); plt.show()

## 5. Size extensivity

For non-interacting subsystems the cluster operators commute, so
$e^{\hat T_A + \hat T_B} = e^{\hat T_A}e^{\hat T_B}$, the wave function
factorises and the energy is additive — at any truncation level.  This is the
property truncated configuration interaction cannot have.

In [ ]:
cc.demo_extensivity()

## 6. Everything against everything

In [ ]:
cc.demo_comparison()

## The full program

Everything above lives in
`BookManybody/BookMaterial/Programs/coupledcluster.py`, which runs as a
script and prints all six demonstrations of the chapter.

In [ ]:
print(open(cc.__file__).read())